# 第 1 周末练习 —— 代码解释助手（OpenRouter + Ollama）

## 练习目标（理念）

为展示你对 **OpenAI 兼容 API**（经 OpenRouter）以及本地 **Ollama** 的熟悉程度，请构建一个小工具：

- **输入**：一段需要解释的代码（或技术问题）
- **输出**：像在和初级工程师说话一样的清晰解释
- **对比**：同一条 system prompt，分别用云端 `openai/gpt-4o-mini`（流式）与本地 `llama3.2:1b`（非流式）回答

这是你在课程期间自己也能天天用的工具：看不懂的代码丢进来问模型。

## 和本课 Day 1 / Day 2 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Chat Completions API | `chat.completions.create(...)` |
| `messages`（system / user） | `SYSTEM_PROMPT` + `get_question_from_code(...)` |
| 流式输出 `stream=True` | OpenRouter 一格边收边 `update_display` |
| OpenAI 兼容客户端 | `OpenAI(base_url=..., api_key=...)` 同时对接 OpenRouter 与 Ollama |
| 本地 Ollama | `http://localhost:11434/v1`，模型 `llama3.2:1b` |

## 怎么跑

1. 从上到下依次运行每个单元格（Shift+Enter）
2. 准备好 `.env`：`OPENROUTER_API_BASE_URL`、`OPENROUTER_API_KEY`；本地需已启动 Ollama 并拉取 `llama3.2:1b`
3. 在构造问题处改示例代码，再分别跑 OpenRouter 流式格与 Llama 格，对比回答风格


In [ ]:
# ========== 导入：把后面要用的工具箱搬进来 ==========

# 导入标准库 os：读环境变量（Environment Variables），例如 OpenRouter 的 base URL 与 API Key
import os
# 从 dotenv 导入 load_dotenv：把 .env 文件里的密钥读进环境变量，避免把密钥写进代码
from dotenv import load_dotenv
# 从 IPython.display 导入展示工具：Markdown 渲染、display 首次显示、update_display 流式刷新同一块区域
from IPython.display import Markdown, display, update_display
# 从 openai 导入 OpenAI 客户端类：既能打 OpenRouter，也能打 Ollama 的 OpenAI 兼容接口
from openai import OpenAI

# 加载 .env；override=True 表示用文件里的值覆盖进程里已有的同名环境变量
load_dotenv(override=True)

# 从环境变量读取 OpenRouter 的 API 根地址（例如 https://openrouter.ai/api/v1）
api_base_url = os.getenv('OPENROUTER_API_BASE_URL')
# 从环境变量读取 OpenRouter 的 API Key（不要打印出来）
api_key = os.getenv('OPENROUTER_API_KEY')

# 两个都拿到才算配置齐全；否则提示无效（打印字符串保持英文原样，可能被脚本/习惯依赖）
if api_base_url and api_key:
  print('Env loaded correctly')
else:
  print('Invalid base url and api key')


In [ ]:
# ========== 常量：模型名与 system prompt 集中写在一处 ==========

# OpenRouter 上的模型路由名：厂商前缀 openai/ + 模型 gpt-4o-mini（字符串必须和平台一致）
MODEL_OPENROUTER = 'openai/gpt-4o-mini'
# 本地 Ollama 模型名：需事先 ollama pull llama3.2:1b；含 tag :1b 表示 1B 参数小模型
MODEL_LLAMA = 'llama3.2:1b'

# system prompt 保留英文：这是发给模型的指令，改译会改变回答风格/行为
SYSTEM_PROMPT = "You are a code assistant that can help user to explain about the code user gave. Respond it with the easy explanation like talking with junior engineer"


In [ ]:
# ========== 客户端：两个 OpenAI 兼容后端，接口形状相同、底座不同 ==========

# OpenRouter 客户端：base_url / api_key 来自上格读到的环境变量
openrouter = OpenAI(
  base_url=api_base_url,
  api_key=api_key,
)

# Ollama 客户端：走本机 OpenAI 兼容端点 /v1；api_key 占位字符串 "ollama"（本地通常不校验）
ollama = OpenAI(
  base_url="http://localhost:11434/v1",
  api_key="ollama",
)


In [ ]:
# ========== 构造 user 问题：把「要解释的代码」包进固定英文模板 ==========

# 入参 code：任意代码片段字符串；返回拼好的 user 提问文本
def get_question_from_code(code):
  # f-string 模板保留英文：这是发给模型的 user prompt，改译会改变行为
  question = f"""
Please explain what this code does and why:
{code}
"""
  # 把拼好的问题返回给调用方，稍后放进 messages 的 user.content
  return question


In [ ]:
# ========== OpenRouter 流式回答：边生成边刷新笔记本里的 Markdown ==========

# 调用 OpenRouter 的 Chat Completions；stream=True 表示持续返回增量 delta，而不是一次整段
stream = openrouter.chat.completions.create(
  model=MODEL_OPENROUTER,
  messages=[
    # system：定「怎么解释」（用上面的 SYSTEM_PROMPT）
    {"role": "system", "content": SYSTEM_PROMPT},
    # user：示例代码是 load_dotenv 这一行；可改成你想解释的任意代码
    {"role": "user", "content": get_question_from_code("from dotenv import load_dotenv")}
  ],
  stream=True,
)

# 累积已收到的全文，供每次 update_display 重新渲染
response = ""
# display_id=True：拿到可刷新的句柄，后面用同一 display_id 原地更新，而不是刷出一堆新格子
display_handle = display(Markdown(""), display_id=True)
# 遍历流式 chunk；每个 chunk 通常带一小段 delta.content
for chunk in stream:
  # or ''：有的 chunk 没有 content（例如结束标记），用空串避免把 None 拼进去
  response += chunk.choices[0].delta.content or ''
  # 用当前累积全文刷新同一块 Markdown 显示区
  update_display(Markdown(response), display_id=display_handle.display_id)


In [ ]:
# ========== 本地 Llama：同一套 messages，非流式一次取完整答案 ==========

# 用 Ollama 兼容客户端调用本地模型；默认非流式，等整段生成完再返回
response = ollama.chat.completions.create(
  model=MODEL_LLAMA,
  messages=[
    # system / user 与上一格相同，便于对比云端与本地风格差异
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": get_question_from_code("from dotenv import load_dotenv")}
  ],
)

# 取出助手最终文本，包成 Markdown 对象（在 notebook 里作为单元格结果展示）
Markdown(response.choices[0].message.content)
